# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Every eligible item (≥100 impressions in March 2026) gets a reason code: `CTR_BELOW_MODEL_EXPECTED` if the model's predicted CTR beats the actual CTR by more than 0.05 percentage points, otherwise `PERFORMING_AS_EXPECTED`.

Flagged items are grouped into three archetypes by average search position, because the fix depends on where the item already sits:
- **high_rank_ignored** (position 1–6) — ranking well, but people aren't clicking → check the title/snippet
- **mid_rank_ignored** (position 7–10) — borderline visibility → check title/snippet and whether the content actually matches the query
- **low_rank_ignored** (position 11+) — not ranking well enough to expect clicks yet → work on ranking signals first

The queue is sorted by score (biggest CTR gap first), then impressions (bigger audience first), so the top of the list is where a fix matters most, soonest.

In [ ]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{userdata.get("HF_TOKEN")}'
);
""")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_query = f"""
WITH agg AS (
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{MARCH_PATH}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
)
SELECT
    a.client_hash_id, a.content_hash_id,
    a.avg_position, a.impressions,
    d.main_intent, d.search_volume, d.category_count,
    a.clicks * 1.0 / NULLIF(a.impressions, 0) AS actual_ctr
FROM agg a
JOIN read_parquet('{CONTENT_PATH}') d ON a.content_hash_id = d.content_hash_id
"""
df = con.sql(feature_query).df().dropna(subset=["actual_ctr", "avg_position", "main_intent",
                                                 "search_volume", "category_count", "impressions"])
numeric_cols = ["avg_position", "impressions", "search_volume", "category_count"]
df[numeric_cols] = df[numeric_cols].astype("float64")

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ["avg_position", "impressions", "search_volume", "category_count"]
categorical_features = ["main_intent"]

pre = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])
deployment_model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
])
deployment_model.fit(df[numeric_features + categorical_features], df["actual_ctr"])

df["predicted_ctr"] = deployment_model.predict(df[numeric_features + categorical_features])
df["score"] = df["predicted_ctr"] - df["actual_ctr"]

def archetype(pos):
    if pos <= 6: return "high_rank_ignored"
    elif pos <= 10: return "mid_rank_ignored"
    else: return "low_rank_ignored"

def action_for(arch):
    return {
        "high_rank_ignored": "review_snippet_title",
        "mid_rank_ignored": "review_snippet_title + content_relevance_check",
        "low_rank_ignored": "improve_ranking_signals",
    }[arch]

df["archetype"] = df["avg_position"].apply(archetype)
df["action"] = df["archetype"].apply(action_for)
df["reason_code"] = df["score"].apply(
    lambda s: "CTR_BELOW_MODEL_EXPECTED" if s > 0.0005 else "PERFORMING_AS_EXPECTED"
)

ranked_queue = df[df["reason_code"] == "CTR_BELOW_MODEL_EXPECTED"].sort_values(
    ["score", "impressions"], ascending=[False, False]
)
print(f"{len(ranked_queue)} of {len(df)} items flagged for action ({len(ranked_queue)/len(df)*100:.1f}%)")
ranked_queue[["content_hash_id", "archetype", "action", "avg_position", "impressions",
              "actual_ctr", "predicted_ctr", "score", "reason_code"]].head(15)

**Result:** 58,480 of 99,197 eligible items flagged for action (**59.0%**). That's a large share, but it lines up with an earlier finding that ~37% of the eligible pool was tied at zero clicks — a big chunk of items are underperforming their expected CTR by construction, not by surprise.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This queue is for FlyRank's SEO team, to decide which content to review first — not to auto-publish changes. It's built entirely on March 2026 GSC performance, a single month's snapshot, so it doesn't account for seasonal swings or one-off traffic spikes.

It stops being valid for:
- Items with under 100 impressions in the month (excluded — too little data to trust the CTR estimate)
- Brand-new content with no performance history yet
- Any client outside this warehouse's dataset

The "predicted CTR" comes from a model trained on the same month it's scoring, so a gap between predicted and actual CTR is a signal to go look, not a guarantee that fixing the title will close it.

In [ ]:
print(f"Eligibility threshold: impressions >= 100, March 2026 only")
print(f"Eligible items after filter: {len(df)}")
print(f"Flagged for action: {len(ranked_queue)} ({len(ranked_queue)/len(df)*100:.1f}%)")
print(f"Performing as expected: {len(df) - len(ranked_queue)} ({(len(df)-len(ranked_queue))/len(df)*100:.1f}%)")

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged item, a person must:
- Read the actual page/snippet, not just trust the archetype label — `main_intent` is inferred per-content, not verified per-item
- Check the item isn't flagged purely because of a data artifact (a tracking gap, a one-day traffic spike inflating impressions)
- Confirm the recommended action fits that client's brand voice before anything gets rewritten

Never automated:
- Auto-publishing a title/snippet rewrite without a human reading it first
- Treating `low_rank_ignored` as "not worth pursuing" — that archetype needs a content/SEO strategy call, not a script
- Treating all zero-click items the same — some are zero because they're genuinely irrelevant, others because they're new or under-indexed, and the fix differs

In [ ]:
zero_click_share = (ranked_queue["actual_ctr"] == 0).mean() * 100
print(f"Share of flagged queue with actual_ctr == 0: {zero_click_share:.1f}%")
print("None of these get auto-published — every flagged item goes through a human review step before any change goes live.")

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain or re-check the model when:
- The next month's MAE moves meaningfully outside the established range (0.00243–0.00277) — a sign the CTR-position relationship shifted, e.g. a Google SERP layout change
- `queue_fraction_pct` jumps sharply month over month (currently 59.0%) — could mean a real shift in client content quality, or a broken upstream join/filter
- A new month's average-position distribution looks noticeably different from March's — the model was validated time-aware (Feb→March, MAE 0.00243 both directions) but hasn't been tested across a bigger time gap or a seasonal boundary yet

In [ ]:
baseline_mae_range = (0.00243, 0.00277)
baseline_queue_fraction_pct = 59.0

def check_for_drift(new_mae, new_queue_fraction_pct, tol_pct=15):
    mae_ok = baseline_mae_range[0] <= new_mae <= baseline_mae_range[1] * 1.1
    fraction_drift = abs(new_queue_fraction_pct - baseline_queue_fraction_pct) / baseline_queue_fraction_pct * 100
    fraction_ok = fraction_drift <= tol_pct
    return {"mae_within_baseline": mae_ok, "queue_fraction_within_tolerance": fraction_ok}

# Re-run this each month with the new numbers to decide whether to retrain
print(check_for_drift(new_mae=0.00268, new_queue_fraction_pct=59.0))

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/` — your paper builds on these files.*

Write the queue to CSV, the run metrics (including permutation importance) to JSON, and the feature-importance chart to PNG, so the Week 8 paper can build directly on these files without re-running the notebook.

In [ ]:
!pip install matplotlib --quiet

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

ranked_queue.to_csv("work/outputs/content_action_playbook.csv", index=False)

X_eval = df[numeric_features + categorical_features]
y_eval = df["actual_ctr"]
X_train_eval, X_test_eval, y_train_eval, y_test_eval = train_test_split(
    X_eval, y_eval, test_size=0.25, random_state=42
)
perm = permutation_importance(deployment_model, X_test_eval, y_test_eval,
                               n_repeats=10, random_state=42, n_jobs=-1)
importance_dict = dict(zip(numeric_features + categorical_features, perm.importances_mean.round(4)))

metrics = {
    "week4_baseline_mae": 0.00277,
    "week5_grouped_split_mae": 0.00268,
    "week5_improvement_pct": 3.2,
    "week6_within_march_mae": 0.00243,
    "week6_time_aware_feb_to_march_mae": 0.00243,
    "week6_time_aware_gap_pct": 0.0,
    "permutation_importance": importance_dict,
    "queue_size": len(ranked_queue),
    "total_eligible_items": len(df),
    "queue_fraction_pct": round(len(ranked_queue) / len(df) * 100, 1),
    "eligibility_threshold_impressions": 100,
    "score_threshold": 0.0005,
}
with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

importance_series = pd.Series(importance_dict).sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
importance_series.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Permutation importance (mean)")
ax.set_title("Feature importance — validated Random Forest")
plt.tight_layout()
plt.savefig("work/figures/feature_importance.png", dpi=150)
plt.show()

**Result:** `avg_position` remains the dominant signal (0.1336), consistent with every prior week — impressions (0.0302), search_volume (0.0288), category_count (0.0243), and main_intent (0.0047) trail well behind. These magnitudes are on the deployment model's own random eval split, not the Week 5 grouped-by-client split — same ranking, different eval setup.

## Self-check

Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.